# Funciones de Transferencia ITAE Óptimas

Función para generar funciones de transferencia con criterio ITAE (Integral of Time-weighted Absolute Error) para sistemas de control con cero error de posición o cero error de velocidad.

In [ ]:
! pip install control
import control as ctrl
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math


#!wget -O ubicarpolos.py https://raw.githubusercontent.com/nebisman/control-material/refs/heads/main/notebooks/ubicarpolos.py
from ubicarpolos import asigne_polos, calcular_itae, tf_to_latex



##  Otros criterios óptimos de diseño polinomial

Además del ITAE, existen otros criterios de optimización para ubicar los polos de un sistema en lazo cerrado.

### Criterios de error integral
| Criterio | Función de costo | Característica |
|:--------:|:----------------:|:--------------:|
| ISE | $\int_0^\infty e^2(t)\,dt$ | Penaliza errores grandes, respuesta rápida, más sobrepaso |
| IAE | $\int_0^\infty \|e(t)\|\,dt$ | Penalización uniforme, balance entre velocidad y sobrepaso |
| ITAE | $\int_0^\infty t\,\|e(t)\|\,dt$ | Penaliza errores tardíos, asentamiento rápido |

### Criterios de forma polinomial
| Criterio | Característica |
|:--------:|:--------------:|
| Butterworth | Respuesta en frecuencia maximalmente plana. Equivale al ISE óptimo para entrada escalón |
| Bessel | Retardo de grupo máximamente plano (fase lineal). Mínima distorsión de forma de onda |
| Binomial | Todos los polos en $s=-\omega$. Polinomio $(s+\omega)^n$. Cero sobrepico |

In [ ]:
from math import factorial, comb as math_comb

def calcular_iae(orden=3, omega=1):
    """IAE óptimo (entrada escalón): penalización uniforme del error."""
    coef = {
        1: [],
        2: [1.414],
        3: [1.90, 2.20],
        4: [2.20, 3.50, 2.80],
        5: [2.80, 5.00, 5.50, 3.40]
    }
    if orden not in coef:
        raise ValueError(f"Orden {orden} no soportado para IAE. Use 1-5.")
    coefs = coef[orden]
    den = [1.0]
    for i, a in enumerate(coefs):
        den.append(a * omega**(i + 1))
    den.append(omega**orden)
    num = [omega**orden]
    return ctrl.tf(num, den)

def calcular_butterworth(orden=3, omega=1):
    """Butterworth (ISE óptimo): polos equiespaciados en semicírculo."""
    polos = []
    for k in range(1, orden + 1):
        theta = np.pi * (2*k + orden - 1) / (2*orden)
        polos.append(omega * np.exp(1j * theta))
    den = np.real(np.poly(polos)).tolist()
    num = [omega**orden]
    return ctrl.tf(num, den)

def calcular_bessel(orden=3, omega=1):
    """Bessel-Thomson: retardo de grupo maximalmente plano (fase lineal)."""
    coefs_raw = np.zeros(orden + 1)
    for k in range(orden + 1):
        coefs_raw[orden - k] = factorial(2*orden - k) / (
            2**(orden - k) * factorial(k) * factorial(orden - k))
    c0 = coefs_raw[-1]
    alpha = c0 ** (1.0 / orden)
    den_norm = [coefs_raw[i] / alpha**i for i in range(orden + 1)]
    den_final = [den_norm[i] * omega**i for i in range(orden + 1)]
    num = [omega**orden]
    return ctrl.tf(num, den_final)



### Tabla comparativa de funciones de transferencia (orden 3, $\omega = 1$)

| Criterio | $T(s)$ |
|:--------:|:------:|
| ITAE | $\dfrac{1}{s^3 + 1.75\,s^2 + 2.15\,s + 1}$ |
| IAE | $\dfrac{1}{s^3 + 1.90\,s^2 + 2.20\,s + 1}$ |
| Butterworth (ISE) | $\dfrac{1}{s^3 + 2\,s^2 + 2\,s + 1}$ |
| Bessel | $\dfrac{1}{s^3 + 2.4329\,s^2 + 2.4662\,s + 1}$ |


In [ ]:
# Comparación de todos los criterios — Orden 3, omega = 1

criterios = {
    'ITAE':       lambda n, w: calcular_itae(n, w, 'p'),
    'IAE':        calcular_iae,
    'Butterworth (ISE)': calcular_butterworth,
    'Bessel':     calcular_bessel,
}

colores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
estilos = ['-', '--', '-.', ':', (0,(3,1,1,1))]

n, w = 3, 1

# Mostrar funciones de transferencia en LaTeX
print(f'Funciones de transferencia — Orden {n}, ω = {w}:\n')
for nombre, func in criterios.items():
    T = func(n, w)
    latex = tf_to_latex(T)
    display(Math(f'\\text{{{nombre}:}}\\quad ' + latex))

# Gráfica comparativa
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for (nombre, func), color, estilo in zip(criterios.items(), colores, estilos):
    T = func(n, w)
    t, y = ctrl.step_response(T)
    ax1.plot(t, y, color=color, linestyle=estilo, linewidth=2, label=nombre)
    # Calcular sobrepaso y tiempo de asentamiento
    overshoot = 100*(max(y) - 1)
    if overshoot <0:
        overshoot =0
    # Tiempo de asentamiento al 2%
    idx_settled = np.where(np.abs(y - 1) > 0.02)[0]
    t_s = t[idx_settled[-1]] if len(idx_settled) > 0 else 0
    ax2.barh(nombre, overshoot, color=color, alpha=0.7)

ax1.axhline(y=1, color='k', linestyle='--', alpha=0.3)
ax1.set_xlabel('Tiempo [s]')
ax1.set_ylabel('Amplitud')
ax1.set_title(f'Respuesta al escalón — Orden {n}, ω = {w}')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(left=0)
ax1.set_ylim(-0.05, 1.25)

ax2.set_xlabel('Sobrepaso [%]')
ax2.set_title('Sobrepaso máximo por criterio')
ax2.grid(True, alpha=0.3, axis='x')
ax2.axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

# Tabla de métricas
print('\n{:<20s} {:>12s} {:>15s} {:>15s}'.format(
    'Criterio', 'Sobrepaso %', 'Tiempo pico', 'Asentamiento 2%'))
print('-'*65)
for nombre, func in criterios.items():
    T = func(n, w)
    t, y = ctrl.step_response(T)
    overshoot = 100*(max(y) - 1)
    t_peak = t[np.argmax(y)]
    idx = np.where(np.abs(y - 1) > 0.02)[0]
    t_s = t[idx[-1]] if len(idx) > 0 else 0
    print(f'{nombre:<20s} {overshoot:>11.2f}% {t_peak:>14.3f}s {t_s:>14.3f}s')